In [ ]:
!pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report
from xgboost import XGBRegressor, XGBClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

In [ ]:
from google.colab import files

print("📤 Upload: Student Budget Survey.csv")
uploaded = files.upload()

In [ ]:
# Load survey
survey_df = pd.read_csv('Student Budget Survey.csv')
print(f"✅ Loaded {len(survey_df)} records")

# Rename columns
survey_df.columns = [
    'Timestamp', 'Academic_Level', 'Funding_Source', 'Monthly_Income',
    'Transport_Cost', 'Financial_Comfort', 'Affordability_Accommodation',
    'Affordability_Food', 'Affordability_Materials', 'Affordability_Transport',
    'Affordability_Social', 'Financial_Assistance_Needed', 'Work_Hours_Per_Week',
    'Additional_Comments'
]

df = survey_df.copy()
df = df.drop(['Timestamp', 'Additional_Comments'], axis=1)

print("✅ Data loaded and cleaned")

In [ ]:
# Convert Income to numeric
def convert_income(income_str):
    if pd.isna(income_str) or income_str == 'Prefer not to say':
        return 25000
    income_map = {
        'Less than LKR 15,000': 12500,
        'LKR 15,000 - 25,000': 20000,
        'LKR 25,001 - 35,000': 30000,
        'LKR 35,001 - 50,000': 42500,
        'LKR 50,001 - 75,000': 62500,
        'More than LKR 75,000': 90000
    }
    return income_map.get(income_str, 25000)

df['Income'] = df['Monthly_Income'].apply(convert_income)

# Convert Transport Cost
df['Transport'] = pd.to_numeric(df['Transport_Cost'], errors='coerce')
df['Transport'].fillna(df['Transport'].median(), inplace=True)

# Convert Work Hours
def convert_hours(hours_str):
    if pd.isna(hours_str):
        return 0
    hours_map = {'0 hours': 0, '1-5 hours': 3, '6-10 hours': 8, '11-15 hours': 13, 
                 '16-20 hours': 18, 'More than 20 hours': 25}
    return hours_map.get(hours_str, 0)

df['Work_Hours'] = df['Work_Hours_Per_Week'].apply(convert_hours)

# Convert Affordability to numeric (1-5)
aff_map = {'Very Affordable': 5, 'Affordable': 4, 'Neutral': 3, 'Unaffordable': 2, 'Very Unaffordable': 1}
df['Aff_Accommodation'] = df['Affordability_Accommodation'].map(aff_map).fillna(3)
df['Aff_Food'] = df['Affordability_Food'].map(aff_map).fillna(3)
df['Aff_Materials'] = df['Affordability_Materials'].map(aff_map).fillna(3)
df['Aff_Transport'] = df['Affordability_Transport'].map(aff_map).fillna(3)
df['Aff_Social'] = df['Affordability_Social'].map(aff_map).fillna(3)

# Financial Comfort (1-5)
df['Comfort'] = pd.to_numeric(df['Financial_Comfort'], errors='coerce').fillna(3)

# Funding sources (binary features)
df['Has_Parental'] = df['Funding_Source'].str.contains('Parental', na=False).astype(int)
df['Has_Job'] = df['Funding_Source'].str.contains('Part-time', na=False).astype(int)
df['Has_Scholarship'] = df['Funding_Source'].str.contains('Scholarship', na=False).astype(int)
df['Has_Loan'] = df['Funding_Source'].str.contains('Loan', na=False).astype(int)

print("✅ Features engineered")

In [ ]:
# Calculate realistic expense components

# Accommodation cost (inverse of affordability)
# If unaffordable (score 1-2), costs are higher % of income
df['Accommodation_Pct'] = 0.35 - (df['Aff_Accommodation'] - 1) * 0.05  # 35% to 15%
df['Accommodation'] = df['Income'] * df['Accommodation_Pct']

# Food cost
df['Food_Pct'] = 0.35 - (df['Aff_Food'] - 1) * 0.05
df['Food'] = df['Income'] * df['Food_Pct']

# Materials cost
df['Materials_Pct'] = 0.10 - (df['Aff_Materials'] - 1) * 0.02
df['Materials'] = df['Income'] * df['Materials_Pct']

# Social/Entertainment
df['Social_Pct'] = 0.10 - (df['Aff_Social'] - 1) * 0.02
df['Social'] = df['Income'] * df['Social_Pct']

# Utilities & Other (based on comfort level)
df['Other'] = df['Income'] * (0.15 - df['Comfort'] * 0.01)

# Total Monthly Expenses (TARGET)
df['Total_Expenses'] = (
    df['Accommodation'] + 
    df['Food'] + 
    df['Transport'] + 
    df['Materials'] + 
    df['Social'] + 
    df['Other']
)

# Add some realistic noise (±5-10%)
np.random.seed(42)
noise = np.random.normal(1.0, 0.08, len(df))
df['Total_Expenses'] = df['Total_Expenses'] * noise

# Ensure expenses don't exceed income by too much (max 110%)
df['Total_Expenses'] = df[['Total_Expenses', 'Income']].apply(
    lambda x: min(x['Total_Expenses'], x['Income'] * 1.1), axis=1
)

print("✅ Target variable created")
print(f"\n📊 Expense Statistics:")
print(f"   Mean: LKR {df['Total_Expenses'].mean():,.0f}")
print(f"   Median: LKR {df['Total_Expenses'].median():,.0f}")
print(f"   Min: LKR {df['Total_Expenses'].min():,.0f}")
print(f"   Max: LKR {df['Total_Expenses'].max():,.0f}")

In [ ]:
# Visualize Income vs Expenses
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(df['Income'], df['Total_Expenses'], alpha=0.5)
plt.plot([df['Income'].min(), df['Income'].max()], 
         [df['Income'].min(), df['Income'].max()], 'r--', label='Break-even')
plt.xlabel('Monthly Income (LKR)')
plt.ylabel('Total Expenses (LKR)')
plt.title('Income vs Expenses')
plt.legend()

plt.subplot(1, 2, 2)
df['Expense_Ratio'] = (df['Total_Expenses'] / df['Income']) * 100
plt.hist(df['Expense_Ratio'], bins=30, edgecolor='black')
plt.xlabel('Expense/Income Ratio (%)')
plt.ylabel('Frequency')
plt.title('Expense Ratio Distribution')
plt.axvline(x=100, color='r', linestyle='--', label='Break-even')
plt.legend()

plt.tight_layout()
plt.show()

print(f"\n💡 Average expense ratio: {df['Expense_Ratio'].mean():.1f}%")

In [ ]:
# Select features
features = [
    'Income', 'Transport', 'Work_Hours', 'Comfort',
    'Aff_Accommodation', 'Aff_Food', 'Aff_Materials', 'Aff_Transport', 'Aff_Social',
    'Has_Parental', 'Has_Job', 'Has_Scholarship', 'Has_Loan'
]

X = df[features].copy()
y_expenses = df['Total_Expenses'].copy()

# Handle missing values
X.fillna(X.median(), inplace=True)

print(f"✅ Features: {len(features)}")
print(f"✅ Samples: {len(X)}")
print(f"\n📋 Feature List:")
for i, feat in enumerate(features, 1):
    print(f"   {i}. {feat}")

In [ ]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_expenses, test_size=0.2, random_state=42
)

print(f"✅ Training: {len(X_train)} samples")
print(f"✅ Testing: {len(X_test)} samples")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Features scaled")

In [ ]:
print("🤖 Training Budget Prediction Models...\n")

results = {}

# Model 1: Random Forest
print("1️⃣ Random Forest...")
rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)
rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
print(f"   R² = {rf_r2:.4f} ({rf_r2*100:.2f}%), MAE = {rf_mae:,.0f}\n")
results['RF'] = {'model': rf, 'r2': rf_r2, 'mae': rf_mae}

# Model 2: Gradient Boosting
print("2️⃣ Gradient Boosting...")
gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=7, random_state=42)
gbr.fit(X_train_scaled, y_train)
gbr_pred = gbr.predict(X_test_scaled)
gbr_r2 = r2_score(y_test, gbr_pred)
gbr_mae = mean_absolute_error(y_test, gbr_pred)
print(f"   R² = {gbr_r2:.4f} ({gbr_r2*100:.2f}%), MAE = {gbr_mae:,.0f}\n")
results['GBR'] = {'model': gbr, 'r2': gbr_r2, 'mae': gbr_mae}

# Model 3: XGBoost
print("3️⃣ XGBoost...")
xgb = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=7, random_state=42, n_jobs=-1)
xgb.fit(X_train_scaled, y_train)
xgb_pred = xgb.predict(X_test_scaled)
xgb_r2 = r2_score(y_test, xgb_pred)
xgb_mae = mean_absolute_error(y_test, xgb_pred)
print(f"   R² = {xgb_r2:.4f} ({xgb_r2*100:.2f}%), MAE = {xgb_mae:,.0f}\n")
results['XGB'] = {'model': xgb, 'r2': xgb_r2, 'mae': xgb_mae}

# Find best model
best_name = max(results, key=lambda x: results[x]['r2'])
best_model = results[best_name]['model']
best_r2 = results[best_name]['r2']
best_mae = results[best_name]['mae']

print("="*60)
print(f"🏆 BEST MODEL: {best_name}")
print(f"   ✅ Accuracy: {best_r2*100:.2f}%")
print(f"   ✅ MAE: LKR {best_mae:,.0f}")
print("="*60)

In [ ]:
# Create risk target (High risk if expenses > 90% income OR comfort <= 2)
df['Savings_Rate'] = ((df['Income'] - df['Total_Expenses']) / df['Income']) * 100
df['Risk'] = ((df['Savings_Rate'] < 10) | (df['Comfort'] <= 2)).astype(int)

print(f"High Risk: {df['Risk'].sum()} students ({df['Risk'].sum()/len(df)*100:.1f}%)")
print(f"Low Risk: {(1-df['Risk']).sum()} students ({(1-df['Risk']).sum()/len(df)*100:.1f}%)")

y_risk = df['Risk'].copy()
_, _, y_train_risk, y_test_risk = train_test_split(X, y_risk, test_size=0.2, random_state=42)

# Train classifier
print("\n🤖 Training Risk Classifier...")
risk_clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
risk_clf.fit(X_train_scaled, y_train_risk)
risk_pred = risk_clf.predict(X_test_scaled)
risk_acc = accuracy_score(y_test_risk, risk_pred)

print(f"\n✅ Risk Classifier Accuracy: {risk_acc*100:.2f}%")
print("\n" + classification_report(y_test_risk, risk_pred, target_names=['Low Risk', 'High Risk']))

In [ ]:
print("💾 Saving models...\n")

# Save budget prediction model
joblib.dump(best_model, 'budget_optimizer_gbr_model_final_optimized.pkl')
print(f"✅ budget_optimizer_gbr_model_final_optimized.pkl")
print(f"   Model: {best_name}")
print(f"   Accuracy: {best_r2*100:.2f}%")

# Save risk classifier
joblib.dump(risk_clf, 'risk_classifier_model_final.pkl')
print(f"\n✅ risk_classifier_model_final.pkl")
print(f"   Accuracy: {risk_acc*100:.2f}%")

# Save scaler
joblib.dump(scaler, 'feature_preprocessor_final.pkl')
print(f"\n✅ feature_preprocessor_final.pkl")

print("\n" + "="*60)
print("🎉 ALL MODELS SAVED!")
print("="*60)
print("\n📥 Download these files and replace in budget_optimizer_files/")